# Learned Primal-Dual for X-ray Scattering Inversion

Implements the Learned Primal-Dual algorithm (Adler & Öktem, IEEE TMI 2018)  
adapted for the inverse problem: **noisy dS₀(q) → ΔPD(R)**

**Forward operator T** and its adjoint **T\*** are built from the DBT kernel  
used in `processing/analyse_mol_dbt.py` — the same transform that generates  
the training targets, so physics is exactly consistent.

```
Input  g  : dS0_all  (nq=101)   — noisy normalized scattering signal
Target f  : dPD_dbt  (nR=1257)  — clean differential pair density

T*: dS0(q) → ΔPD(R)  via DBT  (the inversion step from the pipeline)
T : ΔPD(R) → dS0(q)  via A_dbt (adjoint of T*, fixed matrix)
```

## 0. Imports

In [ ]:
import os
import sys
import glob

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Add processing/ to path so we can import the DBT function directly
PROJECT_ROOT = os.path.dirname(os.getcwd())
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'processing'))
from analyse_mol_dbt import dbt

DEVICE    = 'cuda' if torch.cuda.is_available() else 'cpu'
PD_DIR    = os.path.join(PROJECT_ROOT, 'PD_data')
MODEL_DIR = os.path.join(os.getcwd(), 'checkpoints')
os.makedirs(MODEL_DIR, exist_ok=True)

print(f'Device : {DEVICE}')
print(f'PD_data: {PD_DIR}')

## 1. Build the DBT Operator Matrix

Precompute **A_dbt** (shape `nR × nq`) by applying the DBT to each standard  
basis vector in q-space. This is a one-time cost (~seconds for nq=101).

- **T\*(dS0)** = `dS0 @ A_dbt.T`  — q → R  (the DBT, same as the pipeline)
- **T(ΔPD)**  = `ΔPD @ A_dbt`    — R → q  (consistent adjoint)

In [ ]:
def build_dbt_operator(q, M, verbose=True):
    """
    Build the DBT operator as a dense matrix A_dbt of shape (nR, nq).

    A_dbt[:, j] = DBT(e_j, q, M)  where e_j is the j-th basis vector.

    Returns
    -------
    A_dbt : torch.Tensor  (nR, nq)  float32
    Rgrid : torch.Tensor  (nR,)     float32  [Å]
    """
    nq = len(q)

    # One dummy run to get nR and Rgrid
    e0 = np.zeros(nq); e0[0] = 1.0
    col0, Rgrid_np = dbt(e0, q, M)
    nR = len(col0)

    A = np.zeros((nR, nq), dtype=np.float32)
    for j in range(nq):
        e_j    = np.zeros(nq); e_j[j] = 1.0
        col, _ = dbt(e_j, q, M)
        A[:, j] = col.astype(np.float32)

    if verbose:
        print(f'A_dbt  : {A.shape}  ({A.nbytes/1e6:.1f} MB)')
        print(f'Rgrid  : [{Rgrid_np[0]:.3f}, {Rgrid_np[-1]:.2f}] Å  ({nR} pts)')

    return torch.tensor(A), torch.tensor(Rgrid_np, dtype=torch.float32)


# Load q and M from any available npz file
_ref = np.load(glob.glob(os.path.join(PD_DIR, '*_dbt.npz'))[0])
q_np  = _ref['q']                  # (101,)
M     = len(_ref['Rgrid']) + 1     # 1258

A_dbt, Rgrid = build_dbt_operator(q_np, M)

## 2. Dataset

Each `*_dbt.npz` file contains `nT` trajectory frames. Each frame is one  
training sample: `(dS0_noisy[:, t], dPD_dbt[:, t])`.

**Noise model**: add Gaussian noise to dS0 to simulate experimental shot noise.  
The noise level `sigma` is expressed as a fraction of the signal std.

In [ ]:
class ScatteringDataset(Dataset):
    """
    Loads (dS0, dPD) pairs from all *_dbt.npz files in pd_dir.

    Each trajectory frame is one sample. dS0 is the normalized windowed
    scattering signal (model input); dPD is the differential pair density
    from the DBT inversion (model target).

    Parameters
    ----------
    pd_dir    : directory containing *_dbt.npz files
    noise_std : Gaussian noise std as fraction of per-sample signal std.
                0.0 = no noise (clean), 0.1 = 10% noise. Default 0.05.
    seed      : RNG seed for reproducible noise (default 0).
    """
    def __init__(self, pd_dir, noise_std=0.05, seed=0):
        self.rng       = np.random.default_rng(seed)
        self.noise_std = noise_std
        self.samples   = []   # list of (dS0 (nq,), dPD (nR,)) pairs

        npz_files = sorted(glob.glob(os.path.join(pd_dir, '*_dbt.npz')))
        # Exclude per-mode files — those have a different key layout
        npz_files = [f for f in npz_files if 'permode' not in f]

        for path in npz_files:
            d   = np.load(path)
            dS0 = d['dS0_all'].astype(np.float32)   # (nq, nT)
            dPD = d['dPD_dbt'].astype(np.float32)   # (nR, nT)
            nT  = dS0.shape[1]
            for t in range(nT):
                self.samples.append((dS0[:, t], dPD[:, t]))

        print(f'Loaded {len(npz_files)} file(s)  →  {len(self.samples)} samples')
        print(f'dS0 shape: ({self.samples[0][0].shape[0]},)  '
              f'dPD shape: ({self.samples[0][1].shape[0]},)')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        dS0_clean, dPD = self.samples[idx]

        # Add Gaussian noise proportional to signal std (noise augmentation)
        if self.noise_std > 0:
            sigma  = self.noise_std * float(np.std(dS0_clean))
            noise  = self.rng.standard_normal(dS0_clean.shape).astype(np.float32)
            dS0_noisy = dS0_clean + sigma * noise
        else:
            dS0_noisy = dS0_clean

        return torch.tensor(dS0_noisy), torch.tensor(dPD)


# ── Build dataset and split ────────────────────────────────────────────────
NOISE_STD  = 0.00    # 5% noise — adjust to match expected experimental SNR
BATCH_SIZE = 16
VAL_FRAC   = 0.15

dataset  = ScatteringDataset(PD_DIR, noise_std=NOISE_STD)
n_val    = max(1, int(len(dataset) * VAL_FRAC))
n_train  = len(dataset) - n_val
train_ds, val_ds = random_split(
    dataset, [n_train, n_val],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f'Train: {n_train}  Val: {n_val}  Batches/epoch: {len(train_loader)}')

## 3. Model

**Algorithm 3** from the paper, adapted to 1D signals:

```
f₀ = 0  (nR channels, zero init)
h₀ = 0  (nq channels, zero init)

for i = 1 … I:
    hᵢ  = Λ_dual_i  ( hᵢ₋₁,  T(f²ᵢ₋₁),  g )   # dual update in q-space
    fᵢ  = Λ_primal_i( fᵢ₋₁,  T*(h¹ᵢ)        )   # primal update in R-space

return f¹_I
```

Each `Λ` is a small 3-layer 1D residual CNN. A different network is used per  
iteration — this is the key improvement over basic LISTA.

In [ ]:
class DualNet(nn.Module):
    """
    Dual update CNN — operates in q-space (nq = 101 points).

    Input  : concat(h_prev, T(f²), g)  →  n_dual + 2 channels
    Output : h_new  (residual update)  →  n_dual channels
    """
    def __init__(self, n_dual, n_channels=32, kernel_size=5):
        super().__init__()
        pad = kernel_size // 2
        self.net = nn.Sequential(
            nn.Conv1d(n_dual + 2, n_channels, kernel_size, padding=pad),
            nn.PReLU(n_channels),
            nn.Conv1d(n_channels,  n_channels, kernel_size, padding=pad),
            nn.PReLU(n_channels),
            nn.Conv1d(n_channels,  n_dual,     kernel_size, padding=pad),
        )

    def forward(self, h, Tf, g):
        # h, Tf, g : (batch, channels, nq)
        return h + self.net(torch.cat([h, Tf, g], dim=1))


class PrimalNet(nn.Module):
    """
    Primal update CNN — operates in R-space (nR = 1257 points).

    Input  : concat(f_prev, T*(h¹))  →  n_primal + 1 channels
    Output : f_new (residual update) →  n_primal channels
    """
    def __init__(self, n_primal, n_channels=32, kernel_size=5):
        super().__init__()
        pad = kernel_size // 2
        self.net = nn.Sequential(
            nn.Conv1d(n_primal + 1, n_channels, kernel_size, padding=pad),
            nn.PReLU(n_channels),
            nn.Conv1d(n_channels,   n_channels, kernel_size, padding=pad),
            nn.PReLU(n_channels),
            nn.Conv1d(n_channels,   n_primal,   kernel_size, padding=pad),
        )

    def forward(self, f, Tstar_h):
        # f, Tstar_h : (batch, channels, nR)
        return f + self.net(torch.cat([f, Tstar_h], dim=1))


class LearnedPrimalDual(nn.Module):
    """
    Learned Primal-Dual (Adler & Öktem 2018) for scattering inversion.

    Forward/adjoint operators use the DBT matrix from the pipeline:
      T*(dS0) = dS0 @ A_dbt.T   — DBT inversion   (q → R)
      T(ΔPD)  = ΔPD  @ A_dbt    — adjoint of T*   (R → q)

    Parameters
    ----------
    A_dbt     : (nR, nq) precomputed DBT operator matrix
    n_primal  : memory channels in R-space (paper default: 5)
    n_dual    : memory channels in q-space (paper default: 5)
    n_iter    : unrolled iterations         (paper default: 10)
    n_channels: CNN hidden channels         (paper default: 32)
    """
    def __init__(self, A_dbt,
                 n_primal=5, n_dual=5, n_iter=10, n_channels=32):
        super().__init__()
        self.register_buffer('A_dbt', A_dbt.float())   # (nR, nq) — fixed
        self.n_primal = n_primal
        self.n_dual   = n_dual
        self.n_iter   = n_iter

        # Separate networks per iteration — key improvement over LISTA
        self.dual_nets   = nn.ModuleList(
            [DualNet(n_dual,    n_channels) for _ in range(n_iter)])
        self.primal_nets = nn.ModuleList(
            [PrimalNet(n_primal, n_channels) for _ in range(n_iter)])

    # ── Physics operators ──────────────────────────────────────────────────

    def T(self, f):
        """Forward: ΔPD(R) → dS0(q).  (batch, nR) → (batch, nq)"""
        return f @ self.A_dbt          # (batch,nR) @ (nR,nq) = (batch,nq)

    def T_star(self, h):
        """Adjoint / DBT: dS0(q) → ΔPD(R).  (batch, nq) → (batch, nR)"""
        return h @ self.A_dbt.T        # (batch,nq) @ (nq,nR) = (batch,nR)

    # ── Forward pass ───────────────────────────────────────────────────────

    def forward(self, g):
        """
        g : (batch, nq) — noisy dS0 measurement
        returns (batch, nR) — reconstructed ΔPD
        """
        batch  = g.shape[0]
        nR, nq = self.A_dbt.shape

        # Zero initialisation (paper Section III-C)
        f    = g.new_zeros(batch, self.n_primal, nR)  # primal — R-space
        h    = g.new_zeros(batch, self.n_dual,   nq)  # dual   — q-space
        g_ch = g.unsqueeze(1)                         # (batch, 1, nq)

        for i in range(self.n_iter):
            # Dual update: h_i = Λ^d_i(h, T(f²), g)
            Tf  = self.T(f[:, 1, :]).unsqueeze(1)         # (batch, 1, nq)
            h   = self.dual_nets[i](h, Tf, g_ch)

            # Primal update: f_i = Λ^p_i(f, T*(h¹))
            Tsh = self.T_star(h[:, 0, :]).unsqueeze(1)    # (batch, 1, nR)
            f   = self.primal_nets[i](f, Tsh)

        return f[:, 0, :]                              # (batch, nR)


# ── Instantiate ────────────────────────────────────────────────────────────
model = LearnedPrimalDual(
    A_dbt      = A_dbt,
    n_primal   = 5,
    n_dual     = 5,
    n_iter     = 10,
    n_channels = 32,
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters())
print(f'Parameters: {n_params:,}  (~{n_params/1e3:.0f}k)')
print(f'A_dbt      : {tuple(A_dbt.shape)}  ({A_dbt.nbytes/1e6:.1f} MB)')

## 4. Loss Function

Pure supervised MSE — no additional physics or regularization terms needed.

The LPD architecture already enforces physics at every iteration by calling `T`
and `T*` explicitly. By the time the output is produced, measurement consistency
is baked in by construction, making a separate physics consistency loss redundant.

```
L = MSE(ΔPD_pred, ΔPD_true)
```

In [ ]:
def loss_fn(dpd_pred, dpd_true):
    """
    Supervised MSE between predicted and true ΔPD.

    No physics consistency or sum-rule terms are needed: the LPD architecture
    calls T and T* at every iteration, so measurement consistency is already
    enforced structurally during each forward pass.

    Parameters
    ----------
    dpd_pred : (batch, nR)  model output
    dpd_true : (batch, nR)  clean target from the DBT pipeline

    Returns
    -------
    loss : scalar tensor
    """
    return F.mse_loss(dpd_pred, dpd_true)

## 5. Training

In [ ]:
# ── Hyperparameters ────────────────────────────────────────────────────────
N_EPOCHS  = 200
LR        = 1e-3
GRAD_CLIP = 1.0    # gradient norm clipping (as in paper)

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=N_EPOCHS * len(train_loader)
)

# ── Training loop ──────────────────────────────────────────────────────────
history = {'train_loss': [], 'val_loss': []}

for epoch in range(1, N_EPOCHS + 1):

    # ── Train ──────────────────────────────────────────────────────────────
    model.train()
    train_loss = 0.0

    for dS0_noisy, dPD_true in train_loader:
        dS0_noisy = dS0_noisy.to(DEVICE)
        dPD_true  = dPD_true.to(DEVICE)

        dPD_pred = model(dS0_noisy)
        loss     = loss_fn(dPD_pred, dPD_true)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        scheduler.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    # ── Validate ───────────────────────────────────────────────────────────
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for dS0_noisy, dPD_true in val_loader:
            dS0_noisy = dS0_noisy.to(DEVICE)
            dPD_true  = dPD_true.to(DEVICE)
            val_loss += loss_fn(model(dS0_noisy), dPD_true).item()
    val_loss /= len(val_loader)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)

    if epoch % 10 == 0 or epoch == 1:
        print(f'Epoch {epoch:4d}/{N_EPOCHS}  train={train_loss:.4e}  val={val_loss:.4e}')

# Save checkpoint
ckpt_path = os.path.join(MODEL_DIR, 'lpd_best.pt')
torch.save({'model_state': model.state_dict(), 'history': history}, ckpt_path)
print(f'Saved: {ckpt_path}')

## 6. Training Curves

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogy(history['train_loss'], label='train',      color='steelblue')
ax.semilogy(history['val_loss'],   label='validation', color='firebrick', ls='--')
ax.set_xlabel('Epoch'); ax.set_ylabel('MSE loss')
ax.set_title('Training / validation loss')
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(MODEL_DIR, 'training_curves.png'), dpi=150)
plt.show()

## 7. Evaluation

Compare three reconstructions on a held-out sample:
1. **Classical DBT** — applying T\* directly to the noisy input (no learning)
2. **LPD** — this model
3. **Ground truth** — clean dPD_dbt target

In [ ]:
model.eval()

# Grab one validation batch
dS0_noisy_b, dPD_true_b = next(iter(val_loader))
dS0_noisy_b = dS0_noisy_b.to(DEVICE)
dPD_true_b  = dPD_true_b.to(DEVICE)

with torch.no_grad():
    dPD_lpd         = model(dS0_noisy_b)           # LPD output
    dPD_dbt_classic = model.T_star(dS0_noisy_b)    # classical DBT on noisy input

# Pull first sample to CPU for plotting
R_np    = Rgrid.numpy()
Rm      = R_np < 6.0   # plot up to 6 Å

true_np = dPD_true_b[0].cpu().numpy()
lpd_np  = dPD_lpd[0].cpu().numpy()
dbt_np  = dPD_dbt_classic[0].cpu().numpy()

# ── Figure ────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=False)

for ax, signal, title, color in [
    (axes[0], true_np, 'Ground truth (clean dPD_dbt)', 'black'),
    (axes[1], dbt_np,  'Classical DBT (noisy input)',  'steelblue'),
    (axes[2], lpd_np,  'Learned Primal-Dual',          'firebrick'),
]:
    ax.plot(R_np[Rm], signal[Rm], color=color, lw=1.2)
    ax.axhline(0, color='gray', lw=0.6, ls='--')
    ax.set_xlabel('R (Å)'); ax.set_ylabel('ΔPD (a.u.)')
    ax.set_title(title)

mse_dbt = float(F.mse_loss(dPD_dbt_classic, dPD_true_b).cpu())
mse_lpd = float(F.mse_loss(dPD_lpd,         dPD_true_b).cpu())
fig.suptitle(f'MSE — classical DBT: {mse_dbt:.3e}   LPD: {mse_lpd:.3e}', fontsize=11)
fig.tight_layout()
fig.savefig(os.path.join(MODEL_DIR, 'evaluation.png'), dpi=150)
plt.show()

print(f'Classical DBT MSE : {mse_dbt:.4e}')
print(f'LPD MSE           : {mse_lpd:.4e}')
print(f'Improvement       : {mse_dbt/mse_lpd:.2f}×')

## 8. Reload a Saved Checkpoint

In [ ]:
def load_model(ckpt_path, A_dbt, device='cpu'):
    """Reload a trained LearnedPrimalDual from a checkpoint."""
    m = LearnedPrimalDual(A_dbt=A_dbt).to(device)
    ckpt = torch.load(ckpt_path, map_location=device)
    m.load_state_dict(ckpt['model_state'])
    m.eval()
    return m, ckpt.get('history', {})

# Example:
# model_loaded, history_loaded = load_model(
#     os.path.join(MODEL_DIR, 'lpd_best.pt'), A_dbt, device=DEVICE
# )